### Title: 01_build_nhanes_wweia
### Purpose: Build out the WWEIA dataset by applying exclusion criteria, intergrating covariates and outcomes (insulin resistance [HOMA-IR]), C-reactive protein, and Metabolic Syndrome
### Date: Dec 19, 2025
### Author: Jules Larke

### Import packages

In [ ]:
import pandas as pd
import numpy as np
import os

### Load data for the Ingredients dataset

In [ ]:
# Load ingredients data
wweia_ingredients = pd.read_csv(
    '../../data/00/wweia_dataset/wweia_ingredients_recalls.csv',
    low_memory=False
)

# remove columns we won't be using
wweia_ingredients.drop(columns=['foodcode', 'food_description', 'DMDEDUC3', 'WTINT2YR', 'WTMEC2YR'], inplace=True)

### Apply the exclusion criteria for the variables in the demographic dataset

In [ ]:
# Make a copy of original data
wweia_ingredients = wweia_ingredients.copy()

# first, rename columns so they are more readable
wweia_ingredients.rename(columns={
    'RIDAGEYR': 'Age', 
    'RIAGENDR': 'Sex', 
    'RIDRETH1': 'ethnicity', 
    'INDFMPIR': 'family_pir',
    'DMDEDUC2': 'education'
}, inplace=True)

# subset to exclude participants  <20 years old
wweia_ingredients = wweia_ingredients[wweia_ingredients['Age'] >= 20]

# factorize education levels
wweia_ingredients['education'] = wweia_ingredients['education'].replace([1, 2], 'less than high school graduate')
wweia_ingredients['education'] = wweia_ingredients['education'].replace(3, 'high school graduate or equivalent')
wweia_ingredients['education'] = wweia_ingredients['education'].replace(4, 'some college')
wweia_ingredients['education'] = wweia_ingredients['education'].replace(5, 'college graduate')
wweia_ingredients['education'] = wweia_ingredients['education'].replace([7, 9], 'unknown')

# remove education == unknown
wweia_ingredients = wweia_ingredients[wweia_ingredients['education']!='unknown']

# remove participants that are pregnant and drop column
wweia_ingredients = wweia_ingredients[wweia_ingredients['RIDEXPRG']!=1]
wweia_ingredients.drop(columns='RIDEXPRG', inplace=True)

### The remaining features and targets (insulin resistance, C-reactive protein) need to be downloaded from the web. We will download those data and merge with the participant IDs (SEQN)

### Body Mass Index and Waist Circumference

In [ ]:
# Exam data - Body Measures (BMI: BMXBMI, WC: BMXWAIST)
bmx_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/BMX_C.XPT', format='xport', encoding='utf-8')
bmx_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BMX_D.XPT', format='xport', encoding='utf-8')
bmx_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/BMX_E.XPT', format='xport', encoding='utf-8')
bmx_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BMX_F.XPT', format='xport', encoding='utf-8')
bmx_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/BMX_G.XPT', format='xport', encoding='utf-8')
bmx_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BMX_H.XPT', format='xport', encoding='utf-8')
bmx_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BMX_I.XPT', format='xport', encoding='utf-8')
bmx_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt', format='xport', encoding='utf-8')
bmx_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt', format='xport', encoding='utf-8')

# select features (BMI, WC)
bmx_C = bmx_C[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_D = bmx_D[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_E = bmx_E[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_F = bmx_F[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_G = bmx_G[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_H = bmx_H[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_I = bmx_I[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_P = bmx_P[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_L = bmx_L[['SEQN', 'BMXBMI', 'BMXWAIST']]

bmi = pd.concat([bmx_C, bmx_D, bmx_E, bmx_F, bmx_G, bmx_H, bmx_I, bmx_P, bmx_L])
bmi.rename(columns={'BMXBMI': 'BMI', 'BMXWAIST': 'WC'}, inplace=True)

wweia_ingredients = wweia_ingredients.merge(bmi, on='SEQN', how='left')

### Fasting Glucose and Insulin

In [ ]:
#Lab data - fasting glucose

fg_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L10AM_C.XPT', format='xport', encoding='utf-8')
fg_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/GLU_D.XPT', format='xport', encoding='utf-8')
fg_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/GLU_E.XPT', format='xport', encoding='utf-8')
fg_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/GLU_F.XPT', format='xport', encoding='utf-8')
fg_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/GLU_G.XPT', format='xport', encoding='utf-8')
fg_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/GLU_H.XPT', format='xport', encoding='utf-8')
fg_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/GLU_I.XPT', format='xport', encoding='utf-8')
fg_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GLU.XPT', format='xport', encoding='utf-8')
fg_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GLU_L.XPT', format='xport', encoding='utf-8')

# insulin data for last years
ins_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/INS_H.xpt', format='xport', encoding='utf-8')
ins_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/INS_I.xpt', format='xport', encoding='utf-8')
ins_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_INS.xpt', format='xport', encoding='utf-8')
ins_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/INS_L.xpt', format='xport', encoding='utf-8')
ins_H = ins_H[['SEQN', 'LBXIN']]
ins_I = ins_I[['SEQN', 'LBXIN']]
ins_P = ins_P[['SEQN', 'LBXIN']]
ins_L = ins_L[['SEQN', 'LBXIN']]

# remove those with a sample weight (WTSAF2YR) of zero for fasting glucose, indicates fasting was not completed by the participant
fg_C = fg_C[['SEQN', 'LBXGLU', 'LBXIN', 'WTSAF2YR']]
fg_C = fg_C[fg_C['WTSAF2YR']>1]
fg_C = fg_C.drop(columns='WTSAF2YR')

fg_I = fg_I[['SEQN', 'LBXGLU', 'WTSAF2YR']]
fg_P = fg_P[['SEQN', 'LBXGLU', 'WTSAFPRP']]
fg_L = fg_L[['SEQN', 'LBXGLU', 'WTSAF2YR']]
fg_I = fg_I[fg_I['WTSAF2YR']>1]
fg_P = fg_P[fg_P['WTSAFPRP']>1]
fg_L = fg_L[fg_L['WTSAF2YR']>1]

fg_H = fg_H.merge(ins_H, on='SEQN')
fg_I = fg_I.merge(ins_I, on='SEQN')
fg_P = fg_P.merge(ins_P, on='SEQN')
fg_L = fg_L.merge(ins_L, on='SEQN')

# fasting time was included in these cycles (PHAFSTHR)
fg_D = fg_D[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_E = fg_E[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_F = fg_F[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_G = fg_G[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_H = fg_H[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]

# remove participants with fasting time of less than 8 hours
fg_D = fg_D[fg_D['PHAFSTHR']>=8]
fg_E = fg_E[fg_E['PHAFSTHR']>=8]
fg_F = fg_F[fg_F['PHAFSTHR']>=8]
fg_G = fg_G[fg_G['PHAFSTHR']>=8]
fg_H = fg_H[fg_H['PHAFSTHR']>=8]

# select features for cycles
fg_C = fg_C[['SEQN', 'LBXGLU', 'LBXIN']]
fg_D = fg_D[['SEQN', 'LBXGLU', 'LBXIN']]
fg_E = fg_E[['SEQN', 'LBXGLU', 'LBXIN']]
fg_F = fg_F[['SEQN', 'LBXGLU', 'LBXIN']]
fg_G = fg_G[['SEQN', 'LBXGLU', 'LBXIN']]
fg_H = fg_H[['SEQN', 'LBXGLU', 'LBXIN']]
fg_I = fg_I[['SEQN', 'LBXGLU', 'LBXIN']]
fg_P = fg_P[['SEQN', 'LBXGLU', 'LBXIN']]
fg_L = fg_L[['SEQN', 'LBXGLU', 'LBXIN']]

fg = pd.concat([fg_C, fg_D, fg_E, fg_F, fg_G, fg_H, fg_I, fg_P, fg_L])
fg.rename(columns={'LBXGLU':'fasting_glc_mg_dL', 'LBXIN': 'fasting_ins_uU_mL'}, inplace=True)

### Calibration of fasting glucose and insulin
#### Due to changes in instrumentation over NHANES cycles, equations are provided by CDC (NHANES) to adjust values and compare cycles

In [ ]:
# get the unique participants and their NHANES cycle 
seqn_cycle = wweia_ingredients.drop_duplicates(subset='SEQN')[['SEQN', 'CYCLE']]

# merge the fasting glucose and insulin data with participant cycle data
fg_cycle = fg.merge(seqn_cycle, on='SEQN', how='left')

# drop NAs
fg_cycle = fg_cycle.dropna()

In [ ]:
# function to apply calibrations to the cycle years. This will chain the equations for each cycle where an instrument change occured, successively applying the corrections
# a reference standard was selected for when the last instrument change occurred

def calibrate_nhanes_glucose_insulin(df):
    """
    Calibrate NHANES glucose and insulin measurements from 2003-2023.
    
    GLUCOSE: Uses 2015-2016 (C311) as reference standard
    INSULIN: Uses 2013-2014+ (Tosoh AIA) as reference standard
    
    Parameters:
    df: DataFrame with columns ['SEQN', 'fasting_glc_mg_dL', 'fasting_ins_uU_mL', 'CYCLE']
    
    Returns:
    DataFrame with calibrated glucose and insulin values
    """
    
    df_calibrated = df.copy()
    df_calibrated['glucose_calibrated'] = df_calibrated['fasting_glc_mg_dL'].copy()
    df_calibrated['insulin_calibrated'] = df_calibrated['fasting_ins_uU_mL'].copy()
    
    # =============================================================================
    # GLUCOSE CALIBRATION (Reference: 2015-2016 C311)
    # =============================================================================
    
    for idx, row in df_calibrated.iterrows():
        cycle = row['CYCLE']
        glucose_orig = row['fasting_glc_mg_dL']
        
        if pd.isna(glucose_orig):
            continue
            
        glucose_cal = glucose_orig
        
        if cycle == '03_04':
            # 2003-2004 (Cobas Mira) → 2015-2016 (C311 reference)
            # Step 1: Mira → 911: Y(911) = 0.9815 * X(Mira) + 3.5707
            glucose_cal = 0.9815 * glucose_cal + 3.5707
            
            # Step 2: 911 → ModP: Y(ModP) = X(911) + 1.148
            glucose_cal = glucose_cal + 1.148
            
            # Step 3: ModP/C501 → C311: X(C311) = (Y(C501) - 0.4994) / 0.9776
            glucose_cal = (glucose_cal - 0.4994) / 0.9776
            
        elif cycle == '05_06':
            # 2005-2006 (Hitachi 911) → 2015-2016 (C311 reference)
            # Step 1: 911 → ModP: Y(ModP) = X(911) + 1.148
            glucose_cal = glucose_cal + 1.148
            
            # Step 2: ModP/C501 → C311: X(C311) = (Y(C501) - 0.4994) / 0.9776
            glucose_cal = (glucose_cal - 0.4994) / 0.9776
            
        elif cycle in ['07_08', '09_10', '11_12', '13_14']:
            # 2007-2014 (ModP/C501) → 2015-2016 (C311 reference)
            # ModP/C501 → C311: X(C311) = (Y(C501) - 0.4994) / 0.9776
            glucose_cal = (glucose_cal - 0.4994) / 0.9776
            
        elif cycle in ['15_16', '17_20', '21_22']:
            # 2015-2016+: Same C311 instrument - NO calibration needed
            glucose_cal = glucose_cal
            
        df_calibrated.loc[idx, 'glucose_calibrated'] = glucose_cal
    
    # =============================================================================
    # INSULIN CALIBRATION (Reference: 2013-2014+ Tosoh AIA)
    # =============================================================================
    
    for idx, row in df_calibrated.iterrows():
        cycle = row['CYCLE']
        insulin_orig = row['fasting_ins_uU_mL']
        
        if pd.isna(insulin_orig) or insulin_orig <= 0:
            continue
            
        insulin_cal = insulin_orig
        
        if cycle == '03_04':
            # 2003-2004 (Tosoh) → 2013-2014+ (Tosoh AIA reference)
            # Step 1: 2003-04 Tosoh → Mercodia
            # Y(Mercodia-equivalent) = 0.9501*X(TOSOH) + 1.4890
            insulin_cal = 0.9501 * insulin_cal + 1.4890
            
            # Step 2: Mercodia → Roche
            # Insulin (Roche-equivalent) = 0.8868*Insulin (Mercodia) + [0.0011*Insulin (Mercodia)**2] – 0.0744
            insulin_cal = 0.8868 * insulin_cal + 0.0011 * (insulin_cal ** 2) - 0.0744
            
            # Step 3: Roche → Tosoh AIA
            # Insulin (Tosoh-equivalent) = 10**(1.024*log10(Roche insulin) – 0.0802)
            if insulin_cal > 0:
                insulin_cal = 10 ** (1.024 * np.log10(insulin_cal) - 0.0802)
            
        elif cycle in ['05_06', '07_08']:
            # 2005-2008 (Mercodia) → 2013-2014+ (Tosoh AIA reference)
            # Step 1: Mercodia → Roche
            # Insulin (Roche-equivalent) = 0.8868*Insulin (Mercodia) + [0.0011*Insulin (Mercodia)**2] – 0.0744
            insulin_cal = 0.8868 * insulin_cal + 0.0011 * (insulin_cal ** 2) - 0.0744
            
            # Step 2: Roche → Tosoh AIA
            # Insulin (Tosoh-equivalent) = 10**(1.024*log10(Roche insulin) – 0.0802)
            if insulin_cal > 0:
                insulin_cal = 10 ** (1.024 * np.log10(insulin_cal) - 0.0802)
            
        elif cycle in ['09_10', '11_12']:
            # 2009-2012 (Roche) → 2013-2014+ (Tosoh AIA reference)
            # Roche → Tosoh AIA
            # Insulin (Tosoh-equivalent) = 10**(1.024*log10(Roche insulin) – 0.0802)
            if insulin_cal > 0:
                insulin_cal = 10 ** (1.024 * np.log10(insulin_cal) - 0.0802)
            
        elif cycle in ['13_14', '15_16', '17_20', '21_22']:
            # 2013-2014+: Tosoh AIA - reference standard, no calibration needed
            insulin_cal = insulin_cal
            
        df_calibrated.loc[idx, 'insulin_calibrated'] = insulin_cal
    
    return df_calibrated

def add_calibration_notes(df):
    """Add detailed notes about calibration equations applied"""
    
    calibration_notes = {
        '03_04': 'Glucose: Mira→911→ModP→C311 | Insulin: Tosoh→Mercodia→Roche→Tosoh AIA',
        '05_06': 'Glucose: 911→ModP→C311 | Insulin: Mercodia→Roche→Tosoh AIA', 
        '07_08': 'Glucose: ModP→C311 | Insulin: Mercodia→Roche→Tosoh AIA',
        '09_10': 'Glucose: ModP→C311 | Insulin: Roche→Tosoh AIA',
        '11_12': 'Glucose: ModP→C311 | Insulin: Roche→Tosoh AIA',
        '13_14': 'Glucose: C501→C311 | Insulin: REFERENCE (Tosoh AIA)',
        '15_16': 'Glucose: REFERENCE (C311) | Insulin: REFERENCE (Tosoh AIA)',
        '17_20': 'Glucose: NO CALIBRATION (same C311) | Insulin: REFERENCE (Tosoh AIA)', 
        '21_22': 'Glucose: NO CALIBRATION (same C311) | Insulin: REFERENCE (Tosoh AIA)'
    }
    
    df['calibration_equations'] = df['CYCLE'].map(calibration_notes)
    return df

def print_calibration_summary():
    """Print summary of complete calibration approach"""
    
    print("COMPLETE NHANES GLUCOSE & INSULIN CALIBRATION")
    print("=" * 70)
    print("\nGLUCOSE REFERENCE: 2015-2016 (C311)")
    print("Timeline: Mira → 911 → ModP/C501 → C311")
    print("  Mira→911: Y(911) = 0.9815 * X(Mira) + 3.5707")
    print("  911→ModP: Y(ModP) = X(911) + 1.148")
    print("  C501→C311 inverse: X(C311) = (Y(C501) - 0.4994) / 0.9776")
    print("\nINSULIN REFERENCE: 2013-2014+ (Tosoh AIA)")
    print("Timeline: 2003 Tosoh → Mercodia → Roche → Tosoh AIA")
    print("  2003 Tosoh→Mercodia: Y(Mercodia) = 0.9501*X(Tosoh) + 1.4890")
    print("  Mercodia→Roche: Y(Roche) = 0.8868*X(Mercodia) + 0.0011*X²- 0.0744")
    print("  Roche→Tosoh AIA: Y(Tosoh) = 10**(1.024*log10(X) - 0.0802)")
    print("\nINSULIN METHOD TIMELINE:")
    print("  2003-04: Tosoh method")
    print("  2005-08: Mercodia ELISA")
    print("  2009-12: Roche Elecsys")
    print("  2013+:   Tosoh AIA (reference)")
    print("\nCALIBRATION CHAINS:")
    print("  2003-04: Tosoh→Mercodia→Roche→Tosoh AIA (3 steps)")
    print("  2005-08: Mercodia→Roche→Tosoh AIA (2 steps)")
    print("  2009-12: Roche→Tosoh AIA (1 step)")
    print("  2013+:   Reference (no calibration)")
    print("=" * 70)

# Usage function to apply calibrations to your dataframe
def apply_calibrations(df):
    """
    Apply calibrations to your NHANES dataframe
    
    Parameters:
    df: Your dataframe with columns ['SEQN', 'fasting_glc_mg_dL', 'fasting_ins_uU_mL', 'CYCLE']
    
    Returns:
    Calibrated dataframe saved as glu_ins_cal
    """
    
    # Print calibration summary
    print_calibration_summary()
    print("\n")
    
    # Apply calibrations
    df_calibrated = calibrate_nhanes_glucose_insulin(df)
    df_calibrated = add_calibration_notes(df_calibrated)
    
    # Calculate changes for reference
    df_calibrated['glucose_change'] = (df_calibrated['glucose_calibrated'] - 
                                      df_calibrated['fasting_glc_mg_dL']).round(2)
    
    df_calibrated['insulin_change'] = (df_calibrated['insulin_calibrated'] - 
                                      df_calibrated['fasting_ins_uU_mL']).round(2)
    
    # Save calibrated results to specified dataframe name
    global glu_ins_cal
    glu_ins_cal = df_calibrated
    
    print("CALIBRATION COMPLETE!")
    print(f"Calibrated results saved to 'glu_ins_cal' dataframe")
    print(f"Shape: {glu_ins_cal.shape}")
    print("\nNew columns added:")
    print("  - glucose_calibrated: Calibrated glucose values")
    print("  - insulin_calibrated: Calibrated insulin values") 
    print("  - calibration_equations: Notes on equations applied")
    print("  - glucose_change: Change from original glucose")
    print("  - insulin_change: Change from original insulin")
    
    return glu_ins_cal

In [ ]:
# apply the calibrations to the data
glu_ins_cal = apply_calibrations(fg_cycle)

In [ ]:
# select the calibrated values for fasting glucose and insulin
glu_ins_res = glu_ins_cal[['SEQN', 'glucose_calibrated', 'insulin_calibrated']]

# merge the calibrated data with the ingredients dataset
wweia_ingredients = wweia_ingredients.merge(glu_ins_res, on='SEQN', how='left')

In [ ]:
# create feature to indicate fasting glucose is at level considered diabetic
wweia_ingredients['diabetes_fasting_glc'] = np.nan
wweia_ingredients['diabetes_fasting_glc'] = wweia_ingredients['diabetes_fasting_glc'].astype('object')
mask = wweia_ingredients['glucose_calibrated'].notna()
wweia_ingredients.loc[mask, 'diabetes_fasting_glc'] = np.where(
    wweia_ingredients.loc[mask, 'glucose_calibrated'] >= 126, 
    'yes', 
    'no'
)

### HbA1c (glycated hemoglobin)

In [ ]:
#Lab data - glycohemoglobic (hba1c)

gh_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L10_C.XPT', format='xport', encoding='utf-8')
gh_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/GHB_D.XPT', format='xport', encoding='utf-8')
gh_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/GHB_E.XPT', format='xport', encoding='utf-8')
gh_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/GHB_F.XPT', format='xport', encoding='utf-8')
gh_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/GHB_G.XPT', format='xport', encoding='utf-8')
gh_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/GHB_H.XPT', format='xport', encoding='utf-8')
gh_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/GHB_I.XPT', format='xport', encoding='utf-8')
gh_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GHB.xpt', format='xport', encoding='utf-8')
gh_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.xpt', format='xport', encoding='utf-8')

# select feature
gh_C = gh_C[['SEQN', 'LBXGH']]
gh_D = gh_D[['SEQN', 'LBXGH']]
gh_E = gh_E[['SEQN', 'LBXGH']]
gh_F = gh_F[['SEQN', 'LBXGH']]
gh_G = gh_G[['SEQN', 'LBXGH']]
gh_H = gh_H[['SEQN', 'LBXGH']]
gh_I = gh_I[['SEQN', 'LBXGH']]
gh_P = gh_P[['SEQN', 'LBXGH']]
gh_L = gh_L[['SEQN', 'LBXGH']]

gh = pd.concat([gh_C, gh_D, gh_E, gh_F, gh_G, gh_H, gh_I, gh_P, gh_L])
gh.rename(columns={'LBXGH':'hba1c_percent'}, inplace=True)

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(gh, on='SEQN', how='left')

In [ ]:
# create variable to designate diabetic status based on hba1c clinical cutoff
wweia_ingredients['diabetes_hba1c'] = np.nan
wweia_ingredients['diabetes_hba1c'] = wweia_ingredients['diabetes_hba1c'].astype('object')
mask = wweia_ingredients['hba1c_percent'].notna()
wweia_ingredients.loc[mask, 'diabetes_hba1c'] = np.where(
    wweia_ingredients.loc[mask, 'hba1c_percent'] >= 6.5, 
    'yes', 
    'no'
)

### Questionnaire for taking medications that affect blood glucose

In [ ]:
#Questionnaire data - taking insulin or glucose lowering meds

diq_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/DIQ_C.XPT', format='xport', encoding='utf-8')
diq_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/DIQ_D.XPT', format='xport', encoding='utf-8')
diq_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/DIQ_E.XPT', format='xport', encoding='utf-8')
diq_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/DIQ_F.XPT', format='xport', encoding='utf-8')
diq_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DIQ_G.XPT', format='xport', encoding='utf-8')
diq_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DIQ_H.XPT', format='xport', encoding='utf-8')
diq_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DIQ_I.XPT', format='xport', encoding='utf-8')
diq_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt', format='xport', encoding='utf-8')
diq_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DIQ_L.xpt', format='xport', encoding='utf-8')

# select features
diq_C = diq_C[['SEQN', 'DIQ050', 'DIQ070']]
diq_D = diq_D[['SEQN', 'DIQ050', 'DID070']]
diq_E = diq_E[['SEQN', 'DIQ050', 'DID070']]
diq_F = diq_F[['SEQN', 'DIQ050', 'DIQ070']]
diq_G = diq_G[['SEQN', 'DIQ050', 'DIQ070']]
diq_H = diq_H[['SEQN', 'DIQ050', 'DIQ070']]
diq_I = diq_I[['SEQN', 'DIQ050', 'DIQ070']]
diq_P = diq_P[['SEQN', 'DIQ050', 'DIQ070']]
diq_L = diq_L[['SEQN', 'DIQ050', 'DIQ070']]

# hamonize names (same question)
diq_D.rename(columns={'DID070':'DIQ070'}, inplace=True)
diq_E.rename(columns={'DID070':'DIQ070'}, inplace=True)

diq = pd.concat([diq_C, diq_D, diq_E, diq_F, diq_G, diq_H, diq_I, diq_P, diq_L])
diq.rename(columns={'DIQ050':'taking_insulin', 'DIQ070':'taking_diabetic_pills'}, inplace=True)

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(diq, on='SEQN', how='left')

# recode levels for taking insulin / diabetes meds
wweia_ingredients['taking_insulin'] = wweia_ingredients['taking_insulin'].replace(1, 'yes')
wweia_ingredients['taking_insulin'] = wweia_ingredients['taking_insulin'].replace(2, 'no')
wweia_ingredients['taking_insulin'] = wweia_ingredients['taking_insulin'].replace([7, 9], 'unknown')

wweia_ingredients['taking_diabetic_pills'] = wweia_ingredients['taking_diabetic_pills'].replace(1, 'yes')
wweia_ingredients['taking_diabetic_pills'] = wweia_ingredients['taking_diabetic_pills'].replace(2, 'no')
wweia_ingredients['taking_diabetic_pills'] = wweia_ingredients['taking_diabetic_pills'].replace([7, 9], 'unknown')

### Determine if participant is diabetic based on the following criteria:
1. Fasting blood glucose > 126 mg/dL
2. HbA1c > 6.5%
3. Reported taking insulin or other diabetes medication 

In [ ]:
print(wweia_ingredients['diabetes_fasting_glc'].value_counts())
print(wweia_ingredients['diabetes_fasting_glc'].isna().sum())

In [ ]:
print(wweia_ingredients['diabetes_hba1c'].value_counts())
print(wweia_ingredients['diabetes_hba1c'].isna().sum())

In [ ]:
# create single feature for diabetes (yes/no)
# Initialize with None to handle NAs properly
wweia_ingredients['diabetes'] = np.nan
wweia_ingredients['diabetes'] = wweia_ingredients['diabetes'].astype('object')

# Set to 'yes' if any of the four criteria are 'yes'
wweia_ingredients['diabetes'] = np.where(
    wweia_ingredients['diabetes_fasting_glc'] == 'yes', 
    'yes', 
    wweia_ingredients['diabetes']
)
wweia_ingredients['diabetes'] = np.where(
    wweia_ingredients['hba1c_percent'] == 'yes', 
    'yes', 
    wweia_ingredients['diabetes']
)
wweia_ingredients['diabetes'] = np.where(
    wweia_ingredients['taking_insulin'] == 'yes', 
    'yes', 
    wweia_ingredients['diabetes']
)
wweia_ingredients['diabetes'] = np.where(
    wweia_ingredients['taking_diabetic_pills'] == 'yes', 
    'yes', 
    wweia_ingredients['diabetes']
)

# Set to 'no' only if all four criteria are not NA and none are 'yes'
mask_all_not_na = (
    wweia_ingredients['diabetes_fasting_glc'].notna() &
    wweia_ingredients['hba1c_percent'].notna() &
    wweia_ingredients['taking_insulin'].notna() &
    wweia_ingredients['taking_diabetic_pills'].notna()
)
wweia_ingredients.loc[mask_all_not_na & wweia_ingredients['diabetes'].isna(), 'diabetes'] = 'no'

In [ ]:
# remove participants that have diabetes according to above criteria to remove confounding based on the outcomes of interest: insulin resistance / systemic inflammation / metS
wweia_ingredients = wweia_ingredients[wweia_ingredients['diabetes']!='yes']

# clean up columns
wweia_ingredients = wweia_ingredients.drop(columns=['hba1c_percent', 'taking_insulin', 'taking_diabetic_pills'])

In [ ]:
wweia_ingredients.SEQN.nunique()

### HDL cholesterol

In [ ]:
#Lab data - HDL cholesterol

hdl_B = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/L13_B.XPT', format='xport', encoding='utf-8')
hdl_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L13_C.XPT', format='xport', encoding='utf-8')
hdl_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/HDL_D.XPT', format='xport', encoding='utf-8')
hdl_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/HDL_E.XPT', format='xport', encoding='utf-8')
hdl_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/HDL_F.XPT', format='xport', encoding='utf-8')
hdl_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/HDL_G.XPT', format='xport', encoding='utf-8')
hdl_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/HDL_H.XPT', format='xport', encoding='utf-8')
hdl_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/HDL_I.XPT', format='xport', encoding='utf-8')
hdl_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HDL.XPT', format='xport', encoding='utf-8')
hdl_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HDL_L.XPT', format='xport', encoding='utf-8')

In [ ]:
# select columns for HDL
hdl_B = hdl_B[['SEQN', 'LBDHDL']]
hdl_C = hdl_C[['SEQN', 'LBXHDD']]
hdl_D = hdl_D[['SEQN', 'LBDHDD']]
hdl_E = hdl_E[['SEQN', 'LBDHDD']]
hdl_F = hdl_F[['SEQN', 'LBDHDD']]
hdl_G = hdl_G[['SEQN', 'LBDHDD']]
hdl_H = hdl_H[['SEQN', 'LBDHDD']]
hdl_I = hdl_I[['SEQN', 'LBDHDD']]
hdl_P = hdl_P[['SEQN', 'LBDHDD']]
hdl_L = hdl_L[['SEQN', 'LBDHDD']]

hdl_B.rename(columns={'LBDHDL':'LBDHDD'}, inplace=True)
hdl_C.rename(columns={'LBXHDD':'LBDHDD'}, inplace=True)
hdl = pd.concat([hdl_B, hdl_C, hdl_D, hdl_E, hdl_F, hdl_G, hdl_H, hdl_I, hdl_P, hdl_L])
hdl.rename(columns={'LBDHDD':'hdl_mg_dL'}, inplace=True)
wweia_ingredients = wweia_ingredients.merge(hdl, on='SEQN', how='left')

### Serum Lipids

In [ ]:
#Lab data - serum lipids (triglycerides)

tg_B = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/L13AM_B.XPT', format='xport', encoding='utf-8')
tg_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L13AM_C.XPT', format='xport', encoding='utf-8')
tg_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/TRIGLY_D.XPT', format='xport', encoding='utf-8')
tg_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/TRIGLY_E.XPT', format='xport', encoding='utf-8')
tg_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/TRIGLY_F.XPT', format='xport', encoding='utf-8')
tg_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/TRIGLY_G.XPT', format='xport', encoding='utf-8')
tg_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/TRIGLY_H.XPT', format='xport', encoding='utf-8')
tg_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/TRIGLY_I.XPT', format='xport', encoding='utf-8')
tg_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TRIGLY.XPT', format='xport', encoding='utf-8')
tg_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/TRIGLY_L.XPT', format='xport', encoding='utf-8')

In [ ]:
tg_B = tg_B[['SEQN', 'LBXTR']]
tg_C = tg_C[['SEQN', 'LBXTR']]
tg_D = tg_D[['SEQN', 'LBXTR']]
tg_E = tg_E[['SEQN', 'LBXTR']]
tg_F = tg_F[['SEQN', 'LBXTR']]
tg_G = tg_G[['SEQN', 'LBXTR']]
tg_H = tg_H[['SEQN', 'LBXTR']]
tg_I = tg_I[['SEQN', 'LBXTR']]
tg_P = tg_P[['SEQN', 'LBXTR']]
tg_L = tg_L[['SEQN', 'LBXTLG']]

tg_L = tg_L.rename(columns={'LBXTLG':'LBXTR'})

tg = pd.concat([tg_B, tg_C, tg_D, tg_E, tg_F, tg_G, tg_H, tg_I, tg_P, tg_L])
tg = tg.rename(columns={'LBXTR':'tg_mg_dL'})

In [ ]:
wweia_ingredients = wweia_ingredients.merge(tg, on='SEQN', how='left')

### Hypertension

In [ ]:
#Exam data - blood pressure (hypertension)

bp_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/BPX_C.XPT', format='xport', encoding='utf-8')
bp_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BPX_D.XPT', format='xport', encoding='utf-8')
bp_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/BPX_E.XPT', format='xport', encoding='utf-8')
bp_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BPX_F.XPT', format='xport', encoding='utf-8')
bp_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/BPX_G.XPT', format='xport', encoding='utf-8')
bp_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BPX_H.XPT', format='xport', encoding='utf-8')
bp_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BPX_I.XPT', format='xport', encoding='utf-8')
bp_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPXO.XPT', format='xport', encoding='utf-8')
bp_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPXO_L.XPT', format='xport', encoding='utf-8')

## systolic
# removed the fourth systolic measurment for cycles C-I as cycles P and L only have three measurements
bp_sys_C = bp_C[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_D = bp_D[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_E = bp_E[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_F = bp_F[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_G = bp_G[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_H = bp_H[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_I = bp_I[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_P = bp_P[['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3']]
bp_sys_L = bp_L[['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3']]


# harmonize column names
bp_sys_P = bp_sys_P.rename(columns={'BPXOSY1': 'BPXSY1', 'BPXOSY2': 'BPXSY2', 'BPXOSY3': 'BPXSY3'})
bp_sys_L = bp_sys_L.rename(columns={'BPXOSY1': 'BPXSY1', 'BPXOSY2': 'BPXSY2', 'BPXOSY3': 'BPXSY3'})

# join cycles
bp_sys = pd.concat([bp_sys_C, bp_sys_D, bp_sys_E, bp_sys_F, bp_sys_G, bp_sys_H, bp_sys_I, bp_sys_P, bp_sys_L])
bp_sys.set_index('SEQN', inplace=True)

# compute the arithmetic mean for all measurements ignoring NaNs
bp_sys['sys_mean'] = bp_sys.mean(axis=1, skipna=True)
bp_sys = bp_sys.reset_index()[['SEQN', 'sys_mean']]

# Using International Diabetes Federation cutoffs for hypertension (130 sys OR 85 diastolic)
# designate hypertensive if systolic BP > 130
bp_sys['sys_ht'] = np.nan
bp_sys['sys_ht'] = bp_sys['sys_ht'].astype('object')
bp_sys['sys_ht'] = np.where(bp_sys['sys_mean'] >= 130, 'yes', bp_sys['sys_ht'])
bp_sys['sys_ht'] = np.where(bp_sys['sys_mean'] < 130, 'no', bp_sys['sys_ht'])

# clean up columns
bp_sys.drop(columns='sys_mean', inplace=True)


## diastolic
# removed the fourth diastolic measurment for cycles C-I as cycles P-L only have three measurements
bp_di_C = bp_C[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_D = bp_D[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_E = bp_E[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_F = bp_F[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_G = bp_G[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_H = bp_H[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_I = bp_I[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_P = bp_P[['SEQN', 'BPXODI1', 'BPXODI2', 'BPXODI3']]
bp_di_L = bp_L[['SEQN', 'BPXODI1', 'BPXODI2', 'BPXODI3']]

bp_sys_P = bp_sys_P.rename(columns={'BPXODI1': 'BPXDI1', 'BPXODI1': 'BPXDI2', 'BPXODI1': 'BPXDI3'})
bp_sys_L = bp_sys_L.rename(columns={'BPXODI1': 'BPXDI3', 'BPXODI1': 'BPXDI3', 'BPXODI1': 'BPXDI3'})


bp_di = pd.concat([bp_di_C, bp_di_D, bp_di_E, bp_di_F, bp_di_G, bp_di_H, bp_di_I, bp_di_P, bp_di_L])
bp_di.set_index('SEQN', inplace=True)


# compute the arithmetic mean for all measurements ignoring NaNs
bp_di['di_mean'] = bp_di.mean(axis=1, skipna=True)
bp_di = bp_di.reset_index()[['SEQN', 'di_mean']]

# designate hypertensive if diastolic BP > 85
bp_di['di_ht'] = np.nan
bp_di['di_ht'] = bp_di['di_ht'].astype('object')
bp_di['di_ht'] = np.where(bp_di['di_mean'] >= 85, 'yes', bp_di['di_ht'])
bp_di['di_ht'] = np.where(bp_di['di_mean'] < 85, 'no', bp_di['di_ht'])

# clean up columns
bp_di.drop(columns='di_mean', inplace=True)



In [ ]:
# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(bp_sys, on='SEQN', how='left')
wweia_ingredients = wweia_ingredients.merge(bp_di, on='SEQN', how='left')

In [ ]:
#Questionnaire data - blood pressure (hypertension)

bpq_B = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/BPQ_B.XPT', format='xport', encoding='utf-8')
bpq_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/BPQ_C.XPT', format='xport', encoding='utf-8')
bpq_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BPQ_D.XPT', format='xport', encoding='utf-8')
bpq_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/BPQ_E.XPT', format='xport', encoding='utf-8')
bpq_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BPQ_F.XPT', format='xport', encoding='utf-8')
bpq_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/BPQ_G.XPT', format='xport', encoding='utf-8')
bpq_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BPQ_H.XPT', format='xport', encoding='utf-8')
bpq_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BPQ_I.XPT', format='xport', encoding='utf-8')
bpq_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPQ.xpt', format='xport', encoding='utf-8')
bpq_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPQ_L.xpt', format='xport', encoding='utf-8')

# select feature: are you taking high BP medication?
bpq_B = bpq_B[['SEQN', 'BPQ050A']]
bpq_C = bpq_C[['SEQN', 'BPQ050A']]
bpq_D = bpq_D[['SEQN', 'BPQ050A']]
bpq_E = bpq_E[['SEQN', 'BPQ050A']]
bpq_F = bpq_F[['SEQN', 'BPQ050A']]
bpq_G = bpq_G[['SEQN', 'BPQ050A']]
bpq_H = bpq_H[['SEQN', 'BPQ050A']]
bpq_I = bpq_I[['SEQN', 'BPQ050A']]
bpq_P = bpq_P[['SEQN', 'BPQ050A']]
bpq_L = bpq_L[['SEQN', 'BPQ150']]

# rename the coded variable to match other cycles (both  are Qs asking if taking high BP medication)
bpq_L = bpq_L.rename(columns={'BPQ150':'BPQ050A'})

bpq = pd.concat([bpq_B, bpq_C, bpq_D, bpq_E, bpq_F, bpq_G, bpq_H, bpq_I, bpq_P, bpq_L])

# merge the BPQ data
wweia_ingredients = wweia_ingredients.merge(bpq, on='SEQN', how='left')

In [ ]:
# create varible to consider participants hypertensive if BP is over clinical threshold or taking medication for high BP
wweia_ingredients['hypertension'] = np.where(wweia_ingredients['sys_ht'] == 'yes', 'yes', wweia_ingredients['sys_ht'])
wweia_ingredients['hypertension'] = np.where(wweia_ingredients['di_ht'] == 'yes', 'yes', wweia_ingredients['hypertension'])
wweia_ingredients['hypertension'] = np.where(wweia_ingredients['BPQ050A'] == 1, 'yes', wweia_ingredients['hypertension'])

### CVD / Cancer

In [ ]:
#Questionnaire data - CVD / Cancer

mc_B = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/MCQ_B.XPT', format='xport', encoding='utf-8')
mc_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/MCQ_C.XPT', format='xport', encoding='utf-8')
mc_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/MCQ_D.XPT', format='xport', encoding='utf-8')
mc_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/MCQ_E.XPT', format='xport', encoding='utf-8')
mc_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/MCQ_F.XPT', format='xport', encoding='utf-8')
mc_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/MCQ_G.XPT', format='xport', encoding='utf-8')
mc_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/MCQ_H.XPT', format='xport', encoding='utf-8')
mc_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/MCQ_I.XPT', format='xport', encoding='utf-8')
mc_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_MCQ.XPT', format='xport', encoding='utf-8')
mc_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/MCQ_L.XPT', format='xport', encoding='utf-8')

mc_B = mc_B[['SEQN', 'MCQ160C', 'MCQ220']]
mc_C = mc_C[['SEQN', 'MCQ160C', 'MCQ220']]
mc_D = mc_D[['SEQN', 'MCQ160C', 'MCQ220']]
mc_E = mc_E[['SEQN', 'MCQ160C', 'MCQ220']]
mc_F = mc_F[['SEQN', 'MCQ160C', 'MCQ220']]
mc_G = mc_G[['SEQN', 'MCQ160C', 'MCQ220']]
mc_H = mc_H[['SEQN', 'MCQ160C', 'MCQ220']]
mc_I = mc_I[['SEQN', 'MCQ160C', 'MCQ220']]
mc_P = mc_P[['SEQN', 'MCQ160C', 'MCQ220']]
mc_L = mc_L[['SEQN', 'MCQ160C', 'MCQ220']]

# concat the cycles and merge with the ingredients dataset
mc = pd.concat([mc_B, mc_C, mc_D, mc_E, mc_F, mc_G, mc_H, mc_I, mc_P, mc_L])
wweia_ingredients = wweia_ingredients.merge(mc, on='SEQN', how='left')

# recode 0/1 to no/yes
wweia_ingredients['MCQ160C'] = np.where(wweia_ingredients['MCQ160C'] == 1, 'yes', 'no')
wweia_ingredients['MCQ220'] = np.where(wweia_ingredients['MCQ220'] == 1, 'yes', 'no')

# remove if ever told had CHD
wweia_ingredients = wweia_ingredients[wweia_ingredients['MCQ160C'] != 'yes']

# remove if ever told had Cancer
wweia_ingredients = wweia_ingredients[wweia_ingredients['MCQ220'] != 'yes']

### Smoking status

In [ ]:
# Questionnaire data - Current or ever smoker
smq_B = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/SMQ_B.XPT', format='xport', encoding='utf-8')
smq_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/SMQ_C.XPT', format='xport', encoding='utf-8')
smq_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/SMQ_D.XPT', format='xport', encoding='utf-8')
smq_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/SMQ_E.XPT', format='xport', encoding='utf-8')
smq_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/SMQ_F.XPT', format='xport', encoding='utf-8')
smq_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/SMQ_G.XPT', format='xport', encoding='utf-8')
smq_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/SMQ_H.XPT', format='xport', encoding='utf-8')
smq_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/SMQ_I.XPT', format='xport', encoding='utf-8')
smq_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.xpt', format='xport', encoding='utf-8')
smq_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/SMQ_L.XPT', format='xport', encoding='utf-8')

In [ ]:
smq_B = smq_B[['SEQN', 'SMQ020']]
smq_C = smq_C[['SEQN', 'SMQ020']]
smq_D = smq_D[['SEQN', 'SMQ020']]
smq_E = smq_E[['SEQN', 'SMQ020']]
smq_F = smq_F[['SEQN', 'SMQ020']]
smq_G = smq_G[['SEQN', 'SMQ020']]
smq_H = smq_H[['SEQN', 'SMQ020']]
smq_I = smq_I[['SEQN', 'SMQ020']]
smq_P = smq_P[['SEQN', 'SMQ020']]
smq_L = smq_L[['SEQN', 'SMQ020']]

smq = pd.concat([smq_B, smq_C, smq_D, smq_E, smq_F, smq_G, smq_H, smq_I, smq_P, smq_L])
smq.rename(columns={'SMQ020':'ever_smoker'}, inplace=True)

In [ ]:
# merge the SMQ data
wweia_ingredients = wweia_ingredients.merge(smq, on='SEQN', how='left')

# recode the ever_smoker to yes/no
wweia_ingredients['ever_smoker'] = wweia_ingredients['ever_smoker'].replace(1, 'yes')
wweia_ingredients['ever_smoker'] = wweia_ingredients['ever_smoker'].replace(2, 'no')
wweia_ingredients['ever_smoker'] = wweia_ingredients['ever_smoker'].replace([7, 9], 'unknown')

# remove ever_smoker == unknown
wweia_ingredients = wweia_ingredients[wweia_ingredients['ever_smoker']!='unknown']

### Calculate combined sample weights for dietary intakes

In [ ]:
# count number of cycles in dataset and show what cycles are included
print(f"Number of unique cycles: {wweia_ingredients['CYCLE'].nunique()}")
print(f"Unique cycles are: {wweia_ingredients['CYCLE'].unique()}")

In [ ]:
# calcualte the sample weights for combining cycles
wweia_ingredients['diet_wts_combined'] = np.where(wweia_ingredients['CYCLE'] == '17_20', (3.2/19.2) * wweia_ingredients['diet_wts'], (2/19.2) * wweia_ingredients['diet_wts'])

In [ ]:
# Check shape of total recall data and unique individuals
print(f"Total rows: {len(wweia_ingredients):,}")
print(f"Unique persons: {wweia_ingredients['SEQN'].nunique():,}")

# Sum weights by unique person only
weights_per_person = wweia_ingredients.groupby('SEQN')['diet_wts_combined'].first()
print(f"Sum of weights (unique persons): {weights_per_person.sum():,.0f}")

### Energy intake - exclude participants with extreme caloric intake (<500, >4500 per day)

In [ ]:
# sum calories for each participant
energy = wweia_ingredients.groupby('SEQN')['Energy'].agg("sum")
energy = energy.reset_index()

# exclude participants with <500 or >4500 kcals
energy = energy[energy['Energy']>500]
energy = energy[energy['Energy']<4500]

# select participants with the range of caloric intake
wweia_ingredients = wweia_ingredients[wweia_ingredients['SEQN'].isin(energy['SEQN'])] 

### Select features for calculating HEI scores for Objective 1

In [ ]:
wweia_ingredients.columns

In [ ]:
# select features needed to calculate HEI. This dataset will also be used to calculate specific Pulse intake and Fiber intake (Figs S1, S2)
wweia_hei = wweia_ingredients[['SEQN', 'ingred_code', 'ingred_desc', 'Ingred_consumed_g', 'Energy', 'Fiber, total dietary', 'Fatty acids, total monounsaturated',
       'Fatty acids, total polyunsaturated', 'Fatty acids, total saturated', 'Sodium', 'Age', 'SDMVPSU', 'SDMVSTRA', 'diet_wts_combined']]

In [ ]:
wweia_cov = wweia_ingredients[['SEQN', 'Age', 'BMI', 'Sex', 'ethnicity', 'family_pir', 'education', 'ever_smoker', 'SDMVPSU', 'SDMVSTRA', 'diet_wts_combined']]

In [ ]:
# Set up output directory
os.makedirs('../../data/01/all/', exist_ok=True)

In [ ]:
# save covariates and survey weights (diet) for full cohort
wweia_cov.dropna().drop_duplicates('SEQN').to_csv('../../data/01/all/wweia_cov_wts.csv', index=None)

In [ ]:
wweia_cov.dropna().drop_duplicates('SEQN')

In [ ]:
# Write file
# Not energy adjusted (HEI2020() function from dietaryindex R package will adjust automatically)
wweia_hei.to_csv('/Users/jules.larke/work/project/pulse_update_copy/data/01/all/all_nhanes_hei_v2.txt', sep='\t', index=None)

## Merge with DFG2 data and apply cutoff to exclude participants consuming <80% calories for carbohydrates mappable to DFG2

In [ ]:
# Load mapping of NHANES to DFG2 for MetS subset
dfg2_map = pd.read_csv('../../data/00/wweia_dataset/nhanes_dfg2_matched.csv')

In [ ]:
# calculate the number of calories per ingredient to determine cutoff of percent calories from carbs consumed
wweia_ingredients['carb_cal'] = wweia_ingredients['Carbohydrate'].copy() * 4

In [ ]:
# find matches in WWEIA to DFG2
recall_match = wweia_ingredients[wweia_ingredients['ingred_code'].isin(dfg2_map['ingred_code'])]

In [ ]:
# calculate the percent of calories from carbohydrates consumed based on the foods that match DFG2
kcal_carb_percent_seqn = recall_match.groupby('SEQN')['carb_cal'].agg("sum") / wweia_ingredients.groupby('SEQN')['carb_cal'].agg("sum")

In [ ]:
kcal_carb_percent_seqn = kcal_carb_percent_seqn.reset_index()

In [ ]:
kcal_carb_percent_seqn.carb_cal.hist(bins=50)
print("mean intake of calories from carbs mappable to DFG2 =", kcal_carb_percent_seqn.carb_cal.mean())

In [ ]:
# subset the participants by cutoff of 80% calories from carbs consumed
kcal_carb_percent_80 = kcal_carb_percent_seqn[kcal_carb_percent_seqn['carb_cal']>0.8000]
nhanes_80 = recall_match[recall_match['SEQN'].isin(kcal_carb_percent_80['SEQN'])]

In [ ]:
# drop columns that we don't need
dfg2_merge = dfg2_map.drop(columns=['ingred_desc'])

In [ ]:
# merge wweia data with dfg2 glycan data
dfg_nhanes = nhanes_80.merge(dfg2_merge, on='ingred_code', how='left')

In [ ]:
# Define column groups
person_id = ['SEQN']
covariates = ['diet_wts', 'Sex', 'Age', 'BMI', 'education', 
              'ethnicity', 'family_pir', 'ever_smoker', 'hypertension']
food_ids = ['ingred_code', 'ingred_desc', 'simple_name', 'food_group']
energy = ['Energy']
nutrients = ['Fiber, total dietary']
glycans = [
    'Arabinan', 'Galactan', 'Xyloglucan', 'Xylan', 'Mannan', 'Beta-Glucan', 
    'Chitin', 'Starch', 'Cellulose', 'Arabinoxylan', 'Galactomannan',
    'Free_Fructose', 'Free_Glucose', 'Free_Sucrose', 'Free_Maltose',
    'Free_Lactose(trehalose)', 'Free_Kestose', 'Free_Raffinose',
    'Free_Maltotetraose', 'Free_Stachyose', 'Free_Maltopentaose',
    'Free_Verbascose', 'Free_Maltohexaose',
    'Mono_Glucose', 'Mono_Galactose', 'Mono_Fructose', 'Mono_Xylose', 
    'Mono_Arabinose', 'Mono_Fucose', 'Mono_Rhamnose', 'Mono_GalA', 
    'Mono_Mannose', 'Mono_Ribose'
]

# Step 1: Adjust glycan values from per 100g to actual consumed amount
df_adjusted = dfg_nhanes.copy()

for glycan in glycans:
    # Multiply glycan concentration (per 100g) by amount consumed and divide by 100
    df_adjusted[glycan] = df_adjusted[glycan] * df_adjusted['Ingred_consumed_g'] / 100

# Step 2: Calculate total energy intake per person for the total dietary intake (pre DFG2 map)
energy = wweia_ingredients.groupby('SEQN')['Energy'].agg("sum")
energy = energy.reset_index()
energy = energy[energy['SEQN'].isin(dfg_nhanes['SEQN'])]
energy = energy.rename(columns={'Energy':'Total_energy'})

# Step 3: Merge total energy back to the original dataframe
df_adjusted = df_adjusted.merge(energy, on='SEQN', how='left')

# Step 4: Adjust fiber and glycans per 1000 calories (overwrite existing columns)
nutrients_glycans = nutrients + glycans

for col in nutrients_glycans:
    # Divide by person's total energy and multiply by 1000 (overwrite original column)
    df_adjusted[col] = (df_adjusted[col] / df_adjusted['Total_energy']) * 1000

In [ ]:
# drop columns and save. This data set will be used to calculate the pulse and non-pulse glycan intakes for the whole cohort
df_adjusted.drop(columns=[
'Total_energy',
'Ingred_consumed_g',
'Carbohydrate',
'Fatty acids, total monounsaturated',
'Fatty acids, total polyunsaturated',
'Fatty acids, total saturated',
'Sodium',
'SDMVPSU',
'SDMVSTRA',
'CYCLE',
'WC',
'glucose_calibrated',
'insulin_calibrated',
'diabetes_fasting_glc',
'diabetes_hba1c',
'diabetes',
'hdl_mg_dL',
'tg_mg_dL',
'sys_ht',
'di_ht',
'BPQ050A',
'hypertension',
'MCQ160C',
'MCQ220',
'carb_cal']).to_csv('../../data/01/all/nhanes_dfg2_ingredient_kcal_adjusted.csv', index=None)

In [ ]:
# remove columns we don't need
df_adjusted = df_adjusted.drop(columns=[
'Total_energy',
'Carbohydrate',
'Fatty acids, total monounsaturated',
'Fatty acids, total polyunsaturated',
'Fatty acids, total saturated',
'Sodium',
'diabetes_fasting_glc',
'diabetes_hba1c',
'diabetes',
'sys_ht',
'di_ht',
'BPQ050A',
'MCQ160C',
'MCQ220',
'carb_cal',
'simple_name'])

In [ ]:
# split off covariates to merge back with participant level glycan intakes
covar = df_adjusted[['SEQN', 'Sex', 'Age', 'ethnicity', 'family_pir', 'education', 
            'glucose_calibrated', 'insulin_calibrated',
            'SDMVPSU', 'SDMVSTRA', 'CYCLE', 'BMI', 'WC',
            'hdl_mg_dL', 'tg_mg_dL', 'hypertension', 'ever_smoker',
            'diet_wts', 'diet_wts_combined']]

covar = covar.drop_duplicates(subset='SEQN')

In [ ]:
covar[['SEQN', 'Sex', 'Age', 'ethnicity', 'family_pir', 'education', 
            'SDMVPSU', 'SDMVSTRA', 'BMI', 'hypertension', 'ever_smoker',
            'diet_wts_combined']].to_csv('../../data/01/all/nhanes_cov_survey_wts.csv', index=None)

In [ ]:
# sum glycan intake per person per food
person_food_glycans = df_adjusted.groupby(['SEQN'])[['Fiber, total dietary', 
       'Arabinan', 'Galactan', 'Xyloglucan',
       'Xylan', 'Mannan', 'Beta-Glucan', 'Chitin', 'Starch', 'Cellulose',
       'Arabinoxylan', 'Galactomannan', 'Free_Fructose', 'Free_Glucose',
       'Free_Sucrose', 'Free_Maltose', 'Free_Lactose(trehalose)',
       'Free_Kestose', 'Free_Raffinose', 'Free_Maltotetraose',
       'Free_Stachyose', 'Free_Maltopentaose', 'Free_Verbascose',
       'Free_Maltohexaose', 'Mono_Glucose', 'Mono_Galactose', 'Mono_Fructose',
       'Mono_Xylose', 'Mono_Arabinose', 'Mono_Fucose', 'Mono_Rhamnose',
       'Mono_GalA', 'Mono_Mannose', 'Mono_Ribose']].agg("sum").reset_index()

# merge with the covariates
nhanes_glycans = covar.merge(person_food_glycans, on= 'SEQN')

## Calculate intake of pulses (all inlcuding those not in DFG2)

In [ ]:
# select the features for getting pulse specific foods
pulse_intake = wweia_ingredients[['SEQN', 'ingred_code', 'ingred_desc', 'Ingred_consumed_g', 'Energy']]

# text matching to get the descriptions for pulse foods
pulse_ingred = pulse_intake[pulse_intake['ingred_desc'].str.contains('beans|black bean|blackeye|black-eyed pea|bayo bean|brown bean|chickpea|cowpea|fava bean|kidney bean|lentil|lima bean|mung bean|navy bean|pigeon pea|pink bean|pinto bean|split|white bean|calico|butter bean', case=False)]

# removing descriptions for foods not considered pulses
pulse_ingred = pulse_ingred[~pulse_ingred['ingred_desc'].str.contains('snap|soybeans|sprouted', case=False)]

In [ ]:
# sum the amount of pulses per gram and energy intake for the pulse foods
intake = pulse_ingred.groupby(['SEQN', 'ingred_code', 'ingred_desc'])[['Ingred_consumed_g', 'Energy']].agg("sum")

# write the list of pulse foods consumed for reference
pulse_ingred.drop_duplicates(subset=['ingred_code', 'ingred_desc'])[['ingred_code', 'ingred_desc']].to_csv('../../data/01/all/all_nhanes_pulse_code_desc.csv', index=None)

In [ ]:
# create conversion factor to convert grams to cups based on either raw or cooked (60 vs. 175 respectively)
pulse_ingred['cup_conversion'] = np.where(pulse_ingred['ingred_desc'].str.contains('raw'), 60, 175)

# convert intakes to cup equivalent (cup_eq)
pulse_ingred['cup_eq'] = pulse_ingred['Ingred_consumed_g'] / pulse_ingred['cup_conversion']

# multiple descriptions of individual pulse foods, clean descriptions and unify
pulse_intake = pulse_ingred.replace({'Beans, baked, canned, plain or vegetarian' : 'Baked beans',
       'Beans, black turtle, mature seeds, canned' : 'Black beans',
       'Beans, black, mature seeds, cooked, boiled, without salt' : 'Black beans',
       'Beans, black, mature seeds, raw' : 'Black beans',
       'Beans, great northern, mature seeds, canned' : 'Great Northern beans',
       'Beans, kidney, red, mature seeds, canned, solids and liquids' : 'Kidney beans',
       'Beans, kidney, red, mature seeds, cooked, boiled, without salt' : 'Kidney beans',
       'Beans, kidney, red, mature seeds, raw' : 'Kidney beans',
       'Beans, pink, mature seeds, cooked, boiled, without salt' : 'Pink beans',
       'Beans, pink, mature seeds, raw' : 'Pink beans',
       'Beans, pinto, mature seeds, cooked, boiled, without salt' : 'Pinto beans',
       'Beans, pinto, mature seeds, raw' : 'Pinto beans',
       'Beans, white, mature seeds, cooked, boiled, without salt' : 'White beans',
       'Beans, yellow, mature seeds, cooked, boiled, without salt' : 'Yellow beans',
       'Broadbeans (fava beans), mature seeds, cooked, boiled, without salt' : 'Fava beans',
       'Chickpeas (garbanzo beans, bengal gram), mature seeds, cooked, boiled, without salt' : 'Chickpeas',
       'Chickpeas (garbanzo beans, bengal gram), mature seeds, raw' : 'Chickpeas',
       'Cowpeas (blackeyes), immature seeds, cooked, boiled, drained, without salt' : 'Cowpeas',
       'Lentils, mature seeds, cooked, boiled, without salt' : 'Lentils',
       'Lentils, raw' : 'Lentils',
       'Lima beans, immature seeds, cooked, boiled, drained, without salt' : 'Lima beans',
       'Lima beans, large, mature seeds, cooked, boiled, without salt' : 'Lima beans',
       'Mung beans, mature seeds, cooked, boiled, without salt' : 'Mung beans',
       'Peas, split, mature seeds, cooked, boiled, without salt' : 'Split peas'
       })

# write the unadjusted pulse intakes in cup eq.
pulse_intake = pulse_intake.groupby(['SEQN', 'ingred_desc'])['cup_eq'].agg("sum").reset_index()

In [ ]:
pulse_intake.SEQN.nunique()

In [ ]:
wweia_seqn = wweia_ingredients.drop_duplicates('SEQN')[['SEQN']]

In [ ]:
non_pulse_intake = wweia_seqn[~wweia_seqn['SEQN'].isin(pulse_intake)]

In [ ]:
# Define the 15 pulse foods in the NHANES

PULSE_FOODS = [
    'Pinto beans', 'White beans', 'Black beans', 'Kidney beans',
    'Baked beans', 'Chickpeas', 'Split peas', 'Lentils', 'Mung beans',
    'Lima beans', 'Cowpeas', 'Pink beans', 'Fava beans',
    'Yellow beans', 'Great Northern beans',
]

# Aggregate reported intake: sum cup_eq per (SEQN × food)
# This handles the case where a person has 2+ rows for the same pulse food

pulse_agg = (
    pulse_intake
    .groupby(['SEQN', 'ingred_desc'], as_index=False)['cup_eq']
    .sum()
)

# Build the full participant × pulse-food grid
# Every participant (consumer + non-consumer) gets a row for every pulse food

all_seqn = pd.concat([
    pulse_intake[['SEQN']],
    non_pulse_intake[['SEQN']],
]).drop_duplicates()

full_grid = all_seqn.merge(
    pd.DataFrame({'ingred_desc': PULSE_FOODS}),
    how='cross',            # all combinations
)

# Left-join reported intakes onto the grid (unmatched → NaN → 0)

full_data = full_grid.merge(pulse_agg, on=['SEQN', 'ingred_desc'], how='left')
full_data['cup_eq'] = full_data['cup_eq'].fillna(0)

# Calculate consumer mean, population mean, and SEM per food

def sem(x):
    """Standard error of the mean."""
    n = len(x)
    return x.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan

results_list = []

for food in PULSE_FOODS:
    food_data = full_data.loc[full_data['ingred_desc'] == food, 'cup_eq']

    # Population (all participants, zeros included)
    pop_mean = food_data.mean()
    pop_sem  = sem(food_data)
    n_pop    = len(food_data)

    # Consumers only (cup_eq > 0)
    consumer_data = food_data[food_data > 0]
    n_consumers   = len(consumer_data)
    con_mean      = consumer_data.mean() if n_consumers > 0 else np.nan
    con_sem       = sem(consumer_data)   if n_consumers > 1 else np.nan

    results_list.append({
        'ingred_desc': food,
        'n_consumers':      n_consumers,
        'consumer_mean':    con_mean,
        'consumer_sem':     con_sem,
        'n_population':     n_pop,
        'population_mean':  pop_mean,
        'population_sem':   pop_sem,
    })

results = pd.DataFrame(results_list)

In [ ]:
# save results for plotting later
results.to_csv('../../data/01/all/pulse_consumer_population_mean_intake.csv', index=None)

In [ ]:
# aggregate by participant for determinine DGA-2020 compliance for Pulse intakes by calorie level
pulse_agg_sum = full_data.groupby('SEQN')['cup_eq'].agg("sum")
pulse_agg_sum = pulse_agg_sum.reset_index()

# get the particpant level calorie intake for computing DGA requirement for pulse consumption
p_energy = wweia_ingredients[['SEQN', 'Energy']]

# sum E intake
p_energy = p_energy.groupby('SEQN')['Energy'].agg("sum")

# merge with pulse data
pulse_agg_sum = pulse_agg_sum.merge(p_energy, how='left', on='SEQN') # get kcal intake for determining if participants met DGA for pulse intake


In [ ]:
# what is the percent of people that consume pulses?
print(round((pulse_agg_sum[pulse_agg_sum['cup_eq']>0].shape[0] / (pulse_agg_sum[pulse_agg_sum['cup_eq']>0].shape[0] + pulse_agg_sum[pulse_agg_sum['cup_eq']==0].shape[0]))*100, 2), '% of people consumed any pulses')

In [ ]:
# DGA 2020-2025 Table A3-2: Beans, Peas, Lentils (cup eq/wk) by calorie level
# Calorie levels and their corresponding weekly pulse recommendation
bin_edges = [0, 1599.99, 1799.99, 2199.99, 2599.99, 2999.99, 6000]
weekly_recs = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

pulse_agg_sum['dga_threshold'] = (
    pd.cut(
        pulse_agg_sum['Energy'],
        bins=bin_edges,
        labels=weekly_recs,
        right=True,        # intervals are (left, right]
        include_lowest=True
    )
    .astype(float) / 7
)

# Flag out-of-range
n_missing = pulse_agg_sum['dga_threshold'].isna().sum()
if n_missing > 0:
    print(f"Warning: {n_missing} participants outside energy range (check their Energy values)")
    print(pulse_agg_sum[pulse_agg_sum['dga_threshold'].isna()]['Energy'].describe())

pulse_agg_sum['met_dga'] = np.where(
    pulse_agg_sum['cup_eq'] >= pulse_agg_sum['dga_threshold'],
    'yes', 'no'
)

In [ ]:
# save the dataset
pulse_agg_sum[['SEQN', 'cup_eq', 'met_dga']].to_csv('../../data/01/all/all_nhanes_pulse_dga.csv', index=None)

In [ ]:
# get the average fiber intake from pulse and non-pulse food sources
pulse_fiber = wweia_ingredients[['SEQN', 'ingred_desc', 'Fiber, total dietary', 'Energy']]

In [ ]:
# Define pulse foods

pulse_ingredients = [
    'Beans, pinto, mature seeds, cooked, boiled, without salt',
    'Beans, white, mature seeds, cooked, boiled, without salt',
    'Beans, black, mature seeds, cooked, boiled, without salt',
    'Beans, kidney, red, mature seeds, canned, solids and liquids',
    'Beans, baked, canned, plain or vegetarian',
    'Chickpeas (garbanzo beans, bengal gram), mature seeds, cooked, boiled, without salt',
    'Peas, split, mature seeds, cooked, boiled, without salt',
    'Beans, kidney, red, mature seeds, cooked, boiled, without salt',
    'Beans, black turtle, mature seeds, canned',
    'Lentils, mature seeds, cooked, boiled, without salt',
    'Mung beans, mature seeds, cooked, boiled, without salt',
    'Lima beans, large, mature seeds, cooked, boiled, without salt',
    'Cowpeas (blackeyes), immature seeds, cooked, boiled, drained, without salt',
    'Beans, pink, mature seeds, raw',
    'Beans, pinto, mature seeds, raw',
    'Beans, pink, mature seeds, cooked, boiled, without salt',
    'Lima beans, immature seeds, cooked, boiled, drained, without salt',
    'Chickpeas (garbanzo beans, bengal gram), mature seeds, raw',
    'Beans, kidney, red, mature seeds, raw',
    'Broadbeans (fava beans), mature seeds, cooked, boiled, without salt',
    'Beans, black, mature seeds, raw',
    'Lentils, raw',
    'Beans, yellow, mature seeds, cooked, boiled, without salt',
    'Beans, great northern, mature seeds, canned',
]

# Tag each row as pulse or non-pulse source

pulse_fiber['fiber_source'] = np.where(
    pulse_fiber['ingred_desc'].isin(pulse_ingredients),
    'pulse', 'non_pulse'
)

# Merge in met_dga status
# pulse_agg_sum should contain ALL participants (consumers + non-consumers)
# with met_dga assigned; verify no missing values after merge

pulse_fiber = pulse_fiber.merge(
    pulse_agg_sum[['SEQN', 'met_dga']],
    on='SEQN',
    how='left'
)

n_missing_dga = pulse_fiber['met_dga'].isna().sum()
if n_missing_dga > 0:
    print(f"Warning: {n_missing_dga} rows missing met_dga — check non-consumer SEQNs in pulse_agg_sum")

# Aggregate fiber per person per source

person_fiber = (
    pulse_fiber
    .groupby(['SEQN', 'met_dga', 'fiber_source'])['Fiber, total dietary']
    .sum()
    .reset_index()
)

# Get all unique participants and their met_dga status

all_participants = (
    pulse_agg_sum[['SEQN', 'met_dga']]
    .drop_duplicates()
)

# Expand grid: every participant needs a pulse and non_pulse fiber row
# Non-consumers will have no pulse fiber rows after step 4 — this adds them
# as explicit zeros so population mean denominators are correct

full_grid = all_participants.merge(
    pd.DataFrame({'fiber_source': ['pulse', 'non_pulse']}),
    how='cross'
)

person_fiber_pop = (
    full_grid
    .merge(person_fiber, on=['SEQN', 'met_dga', 'fiber_source'], how='left')
)
person_fiber_pop['Fiber, total dietary'] = person_fiber_pop['Fiber, total dietary'].fillna(0)

# Verify zero-fill counts look right
print("\nZero-filled pulse fiber rows (should equal number of non-consumers):")
print(
    person_fiber_pop
    .loc[person_fiber_pop['fiber_source'] == 'pulse']
    .groupby('met_dga')['Fiber, total dietary']
    .apply(lambda x: (x == 0).sum())
    .rename('n_zero_pulse_fiber')
)

# Compute population mean and SEM per met_dga group X fiber source

def sem(x):
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

fiber_summary = (
    person_fiber_pop
    .groupby(['met_dga', 'fiber_source'])['Fiber, total dietary']
    .agg(
        n='count',
        population_mean='mean',
        population_sem=sem
    )
    .reset_index()
)

# Total fiber per person (pulse + non-pulse combined)

person_fiber_total = (
    person_fiber_pop
    .groupby(['SEQN', 'met_dga'])['Fiber, total dietary']
    .sum()
    .reset_index()
)

fiber_total_summary = (
    person_fiber_total
    .groupby('met_dga')['Fiber, total dietary']
    .agg(
        n='count',
        population_mean='mean',
        population_sem=sem
    )
    .reset_index()
    .assign(fiber_source='total')
)

fiber_results = (
    pd.concat([fiber_summary, fiber_total_summary], ignore_index=True)
    .sort_values(['met_dga', 'fiber_source'])
)

In [ ]:
person_fiber.groupby(['SEQN', 'met_dga'])['Fiber, total dietary'].agg("sum").to_csv('../../data/01/all/fiber_intake_dga.csv')

In [ ]:
fiber_results.to_csv('../../data/01/all/pulse_fiber_dfg_source_means.csv', index=None)

### Get the metabolic health outcome datasets

### C-reactive protein

In [ ]:
#Lab data - CRP (No CRP data collected from 2011-2014)

crp_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L11_C.XPT', format='xport', encoding='utf-8')
crp_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/CRP_D.XPT', format='xport', encoding='utf-8')
crp_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/CRP_E.XPT', format='xport', encoding='utf-8')
crp_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/CRP_F.XPT', format='xport', encoding='utf-8')
crp_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/HSCRP_I.XPT', format='xport', encoding='utf-8')
crp_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HSCRP.XPT', format='xport', encoding='utf-8')
crp_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HSCRP_L.xpt', format='xport', encoding='utf-8')

crp_I.rename(columns={'LBXHSCRP':'LBXCRP'}, inplace=True)
crp_P.rename(columns={'LBXHSCRP':'LBXCRP'}, inplace=True)
crp_L.rename(columns={'LBXHSCRP':'LBXCRP'}, inplace=True)
crp_L = crp_L.drop(columns='WTPH2YR')

crp_C = crp_C[['SEQN', 'LBXCRP']]
crp_D = crp_D[['SEQN', 'LBXCRP']]
crp_E = crp_E[['SEQN', 'LBXCRP']]
crp_F = crp_F[['SEQN', 'LBXCRP']]
crp_I = crp_I[['SEQN', 'LBXCRP']]
crp_P = crp_P[['SEQN', 'LBXCRP']]
crp_L = crp_L[['SEQN', 'LBXCRP']]

# these cycles measured in units mg/dL, need to convert
crp_convert = pd.concat([crp_C, crp_D, crp_E, crp_F])

# convert from mg/dL to mg/L
crp_convert['LBXCRP'] = crp_convert['LBXCRP'] * 10 

# join converted data
crp = pd.concat([crp_convert, crp_I, crp_P, crp_L])

# drop NAs
crp = crp.dropna()

In [ ]:
# rename CRP
crp.rename(columns={'LBXCRP':'crp'},inplace=True)

# merge the CRP data with participant cycle data
crp_cycle = crp.merge(seqn_cycle, on='SEQN', how='left')

# drop NAs
crp_cycle = crp_cycle.dropna()

#### Calibrate CRP data

In [ ]:
# just as we have done with fasting glucose and insulin, we need to calibrate CRP data due to changes in instrumentation over NHANES cycles

def calibrate_nhanes_crp(df):
    """
    Calibrate NHANES C-reactive protein measurements from 2015-2023.
    
    CRP: Uses 2017-2020 (Cobas 6000) as reference standard
    
    Parameters:
    df: DataFrame with columns ['SEQN', 'crp', 'CYCLE']
    
    Returns:
    DataFrame with calibrated CRP values
    """
    
    df_calibrated = df.copy()
    df_calibrated['crp_calibrated'] = df_calibrated['crp'].copy()
    
    # =============================================================================
    # CRP CALIBRATION (Reference: 2017-2020 Cobas 6000)
    # =============================================================================
    
    # Instrument timeline:
    # 2015-16: DxC 660i → 2017-20: Cobas 6000 (reference) → 2021-23: Cobas 8000
    
    for idx, row in df_calibrated.iterrows():
        cycle = row['CYCLE']
        crp_orig = row['crp']
        
        if pd.isna(crp_orig):
            continue
            
        crp_cal = crp_orig
        
        if cycle == '15_16':
            # 2015-2016 (DxC 660i) → 2017-2020 (Cobas 6000 reference)
            # Given: Y(DxC 660i) = 1.150 * X(Cobas 6000) – 0.3397
            # Inverse: X(Cobas 6000) = (Y(DxC 660i) + 0.3397) / 1.150
            # Note: Equation only applicable for Cobas 6000 values ≤ 20 mg/L
            crp_cal = (crp_cal + 0.3397) / 1.150
            
        elif cycle == '17_20':
            # 2017-2020 (Cobas 6000): Reference standard - NO calibration needed
            crp_cal = crp_cal
            
        elif cycle == '21_22':
            # 2021-2023 (Cobas 8000) → 2017-2020 (Cobas 6000 reference)
            # Given: Forward: Cobas 8000 = -0.08653 + 0.9672 * (Cobas 6000)
            # Inverse: Cobas 6000 = (Cobas 8000 + 0.08653) / 0.9672
            crp_cal = (crp_cal + 0.08653) / 0.9672
            
        df_calibrated.loc[idx, 'crp_calibrated'] = crp_cal
    
    return df_calibrated

def add_crp_calibration_notes(df):
    """Add detailed notes about CRP calibration equations applied"""
    
    calibration_notes = {
        '15_16': 'CRP: DxC 660i→Cobas 6000: (x + 0.3397) / 1.150 [≤20 mg/L only]',
        '17_20': 'CRP: REFERENCE (Cobas 6000)',
        '21_22': 'CRP: Cobas 8000→Cobas 6000: (x + 0.08653) / 0.9672'
    }
    
    df['crp_calibration_equations'] = df['CYCLE'].map(calibration_notes)
    return df

def print_crp_calibration_summary():
    """Print summary of CRP calibration approach"""
    
    print("NHANES C-REACTIVE PROTEIN CALIBRATION")
    print("=" * 60)
    print("\nCRP REFERENCE: 2017-2020 (Cobas 6000)")
    print("Instrument timeline: DxC 660i → Cobas 6000 → Cobas 8000")
    print("\nCalibration equations (converting TO 2017-2020 reference):")
    print("  2015-16 (DxC 660i):")
    print("    Given: Y(DxC 660i) = 1.150 * X(Cobas 6000) – 0.3397")
    print("    Applied: X(Cobas 6000) = (Y(DxC 660i) + 0.3397) / 1.150")
    print("    Note: Only valid for Cobas 6000 values ≤ 20 mg/L")
    print("\n  2021-23 (Cobas 8000):")
    print("    Given: Cobas 8000 = -0.08653 + 0.9672 * (Cobas 6000)")
    print("    Applied: Cobas 6000 = (Cobas 8000 + 0.08653) / 0.9672")
    print("\nCALIBRATION BY CYCLE:")
    print("  2015-16: DxC 660i → Cobas 6000 (with ≤20 mg/L limitation)")
    print("  2017-20: REFERENCE (no calibration)")
    print("  2021-23: Cobas 8000 → Cobas 6000")
    print("\n WARNING: 2015-16 calibration only valid for values ≤20 mg/L")
    print("=" * 60)

def apply_crp_calibrations(df):
    """
    Apply CRP calibrations to your NHANES dataframe
    
    Parameters:
    df: Your dataframe with columns ['SEQN', 'crp', 'CYCLE']
    
    Returns:
    Calibrated dataframe saved as crp_cal
    """
    
    # Print calibration summary
    print_crp_calibration_summary()
    print("\n")
    
    # Check for high CRP values in 2015-16 that exceed calibration limit
    if '15_16' in df['CYCLE'].values:
        high_crp_15_16 = df[(df['CYCLE'] == '15_16') & (df['crp'] > 20)]['crp']
        if len(high_crp_15_16) > 0:
            print(f"   WARNING: Found {len(high_crp_15_16)} CRP values > 20 mg/L in 2015-16 cycle")
            print(f"    Max value: {high_crp_15_16.max():.2f} mg/L")
            print(f"    Calibration equation only valid for ≤20 mg/L")
            print(f"    Consider excluding these values or using alternative approach\n")
    
    # Apply calibrations
    df_calibrated = calibrate_nhanes_crp(df)
    df_calibrated = add_crp_calibration_notes(df_calibrated)
    
    # Calculate changes for reference
    df_calibrated['crp_change'] = (df_calibrated['crp_calibrated'] - 
                                   df_calibrated['crp']).round(4)
    
    # Save calibrated results to specified dataframe name
    global crp_cal
    crp_cal = df_calibrated
    
    print("CRP CALIBRATION COMPLETE!")
    print(f"Calibrated results saved to 'crp_cal' dataframe")
    print(f"Shape: {crp_cal.shape}")
    print("\nNew columns added:")
    print("  - crp_calibrated: Calibrated CRP values (mg/L)")
    print("  - crp_calibration_equations: Notes on equations applied")
    print("  - crp_change: Change from original CRP")
    
    # Summary statistics
    print(f"\nCALIBRATION IMPACT SUMMARY:")
    for cycle in df_calibrated['CYCLE'].unique():
        cycle_data = df_calibrated[df_calibrated['CYCLE'] == cycle]
        if not cycle_data['crp_change'].isna().all():
            mean_change = cycle_data['crp_change'].mean()
            print(f"  {cycle}: Mean change = {mean_change:.4f} mg/L")
    
    return crp_cal

In [ ]:
# apply calibrations
crp_cal = apply_crp_calibrations(crp_cycle)

In [ ]:
# select columns
crp_cal = crp_cal[['SEQN', 'crp_calibrated']]

# rename
crp_cal = crp_cal.rename(columns={'crp_calibrated':'crp'})

### Split the CRP data into its own dataframe

In [ ]:
crp_wweia = nhanes_glycans[['SEQN', 'diet_wts', 'Sex', 'Age', 'BMI', 'education',
                    'ethnicity', 'family_pir', 'ever_smoker', 'CYCLE', 'Fiber, total dietary', 'Arabinan', 'Galactan',
                    'Xyloglucan', 'Xylan', 'Mannan', 'Beta-Glucan', 'Chitin', 'Starch',
                    'Cellulose', 'Arabinoxylan', 'Galactomannan', 'Free_Fructose',
                    'Free_Glucose', 'Free_Sucrose', 'Free_Maltose',
                    'Free_Lactose(trehalose)', 'Free_Kestose', 'Free_Raffinose',
                    'Free_Maltotetraose', 'Free_Stachyose', 'Free_Maltopentaose',
                    'Free_Verbascose', 'Free_Maltohexaose', 'Mono_Glucose',
                    'Mono_Galactose', 'Mono_Fructose', 'Mono_Xylose', 'Mono_Arabinose',
                    'Mono_Fucose', 'Mono_Rhamnose', 'Mono_GalA', 'Mono_Mannose',
                    'Mono_Ribose']]

In [ ]:
crp_wweia = crp_wweia.reset_index()

In [ ]:
# merge with ingredients dataset
crp_subset = crp_wweia.merge(crp_cal, on='SEQN', how='left')

# cutoff to remove very high level of CRP which indicates acute phase response (infection)
crp_subset = crp_subset[crp_subset['crp']<=10]

In [ ]:
# calulate the diet weights for 7 cycles
crp_subset['diet_wts_combined'] = np.where(crp_subset['CYCLE'] == '17_20', (3.2/15.2) * crp_subset['diet_wts'], (2/14) * crp_subset['diet_wts'])

In [ ]:
# remove participants with NAs for CRP
crp_subset = crp_subset.dropna(subset='crp')

In [ ]:
# drop the diet_wts from the original dataframe (only kept to calculate for CRP with the lower number of cycles)
nhanes_glycans = nhanes_glycans.drop(columns='diet_wts')

### Check missing values for CRP subset

In [ ]:
# check covariates for NAs
crp_subset[['Age', 'Sex', 'BMI', 'ever_smoker', 'education', 'ethnicity', 'family_pir']].isna().sum()

In [ ]:
crp_subset = crp_subset.dropna(subset=['Age', 'Sex', 'BMI', 'ever_smoker', 'education', 'ethnicity', 'family_pir'])

In [ ]:
#crp_subset = crp_subset.drop(columns=['diet_wts', 'CYCLE'])

### create a binary class for the outcome, defined as less than 1 mg/L or greater than 3 mg/L

In [ ]:
# target
crp_target = crp_subset[['SEQN', 'crp']]

In [ ]:

crp_target = crp_target.copy()

crp_target['crp_class'] = np.where(crp_target['crp'] > 3, 1, 2)
crp_target['crp_class'] = np.where(crp_target['crp'] < 1, 0, crp_target['crp_class'])

# select PID and remove those with CRP between 1-3 mg/L
crp_target = crp_target[['SEQN', 'crp_class']]
crp_target = crp_target[crp_target['crp_class']!=2]

#### save CRP datasets

In [ ]:
# make the directory
os.makedirs('../../data/01/crp/', exist_ok=True)

crp_target.to_csv('../../data/01/crp/crp_target.tsv', sep='\t', index=None)

In [ ]:
# match participants with glycan intake with CRP outcome data
crp_subset = crp_subset[crp_subset['SEQN'].isin(crp_target['SEQN'])]

# drop target column
crp_subset = crp_subset.drop(columns='crp')

# select columns and save
crp_subset[['SEQN', 'Sex', 'Age', 'BMI', 'education', 'ethnicity', 'family_pir', 
       'ever_smoker', 'diet_wts_combined', 'Fiber, total dietary', 'Arabinan', 'Galactan',
       'Xyloglucan', 'Xylan', 'Mannan', 'Beta-Glucan', 'Chitin', 'Starch',
       'Cellulose', 'Arabinoxylan', 'Galactomannan', 'Free_Fructose',
       'Free_Glucose', 'Free_Sucrose', 'Free_Maltose',
       'Free_Lactose(trehalose)', 'Free_Kestose', 'Free_Raffinose',
       'Free_Maltotetraose', 'Free_Stachyose', 'Free_Maltopentaose',
       'Free_Verbascose', 'Free_Maltohexaose', 'Mono_Glucose',
       'Mono_Galactose', 'Mono_Fructose', 'Mono_Xylose', 'Mono_Arabinose',
       'Mono_Fucose', 'Mono_Rhamnose', 'Mono_GalA', 'Mono_Mannose',
       'Mono_Ribose']].to_csv('../../data/01/crp/crp_dfg2_intake.tsv', sep='\t', index=None)

### Calculate HOMA-IR

In [ ]:
# select columns
homa_wweia = nhanes_glycans[['SEQN', 'diet_wts_combined', 'Sex', 'Age', 'BMI', 'education', 'insulin_calibrated', 'glucose_calibrated',
                    'ethnicity', 'family_pir', 'ever_smoker', 'CYCLE', 'Fiber, total dietary', 'Arabinan', 'Galactan',
                    'Xyloglucan', 'Xylan', 'Mannan', 'Beta-Glucan', 'Chitin', 'Starch',
                    'Cellulose', 'Arabinoxylan', 'Galactomannan', 'Free_Fructose',
                    'Free_Glucose', 'Free_Sucrose', 'Free_Maltose',
                    'Free_Lactose(trehalose)', 'Free_Kestose', 'Free_Raffinose',
                    'Free_Maltotetraose', 'Free_Stachyose', 'Free_Maltopentaose',
                    'Free_Verbascose', 'Free_Maltohexaose', 'Mono_Glucose',
                    'Mono_Galactose', 'Mono_Fructose', 'Mono_Xylose', 'Mono_Arabinose',
                    'Mono_Fucose', 'Mono_Rhamnose', 'Mono_GalA', 'Mono_Mannose',
                    'Mono_Ribose']]

# make a copy
homa_wweia = homa_wweia.copy()

In [ ]:
# homa-ir calculation:
homa_wweia['homa_ir'] = homa_wweia['insulin_calibrated'] * homa_wweia['glucose_calibrated'] / 405

# drop NAs for homa-ir
homa_wweia = homa_wweia.dropna(subset='homa_ir')

# clinical threshold to determine if insulin resistant based on homa ir is >= 2.5
homa_wweia['ir'] = np.where(homa_wweia['homa_ir'] >= 2.5, 1, 0)

# clean up columns
homa_wweia = homa_wweia.drop(columns=['insulin_calibrated', 'glucose_calibrated', 'homa_ir'])

In [ ]:
# check NAs for covariates
homa_wweia[['Age', 'Sex', 'BMI', 'ever_smoker', 'education', 'ethnicity', 'family_pir']].isna().sum()

In [ ]:
# dropping NAs (BMI, family_pir)
homa_wweia = homa_wweia.dropna()

In [ ]:
# split off target
ir = homa_wweia[['SEQN', 'ir']]

In [ ]:
# drop target column
homa_wweia = homa_wweia.drop(columns='ir')

### Generate the IR outcome dataset

In [ ]:
# make the directory
os.makedirs('../../data/01/ir/', exist_ok=True)

# save the ir outcome
ir.to_csv('../../data/01/ir/ir_target.tsv', sep='\t', index=None)

# save the ir target features
homa_wweia.to_csv('../../data/01/ir/ir_dfg2_intake.tsv', sep='\t', index=None)

### Metabolic Syndrome

In [ ]:
mets_wweia = nhanes_glycans[['SEQN', 'Sex', 'Age', 'ethnicity', 'family_pir', 'education',
       'glucose_calibrated', 'SDMVPSU', 'SDMVSTRA',
       'BMI', 'WC', 'hdl_mg_dL', 'tg_mg_dL', 'hypertension',
       'ever_smoker', 'diet_wts_combined', 'Fiber, total dietary',
       'Arabinan', 'Galactan', 'Xyloglucan', 'Xylan', 'Mannan', 'Beta-Glucan',
       'Chitin', 'Starch', 'Cellulose', 'Arabinoxylan', 'Galactomannan',
       'Free_Fructose', 'Free_Glucose', 'Free_Sucrose', 'Free_Maltose',
       'Free_Lactose(trehalose)', 'Free_Kestose', 'Free_Raffinose',
       'Free_Maltotetraose', 'Free_Stachyose', 'Free_Maltopentaose',
       'Free_Verbascose', 'Free_Maltohexaose', 'Mono_Glucose',
       'Mono_Galactose', 'Mono_Fructose', 'Mono_Xylose', 'Mono_Arabinose',
       'Mono_Fucose', 'Mono_Rhamnose', 'Mono_GalA', 'Mono_Mannose',
       'Mono_Ribose']]

In [ ]:
# check NAs for the covariates, less than 10% NAs -> drop NAs for complete case analysis
print(mets_wweia[['Age', 'Sex', 'BMI', 'ever_smoker', 'education', 'ethnicity', 'family_pir']].isna().sum())

# dropping NAs
mets_wweia = mets_wweia.dropna(subset=['Age', 'Sex', 'BMI', 'ever_smoker', 'education', 'ethnicity', 'family_pir'])

In [ ]:
# split the features used to determine MetS
mets_criteria = mets_wweia[['SEQN', 'Sex', 'glucose_calibrated', 'WC', 'hdl_mg_dL', 'tg_mg_dL', 'hypertension']]

In [ ]:
# drop NAs for features we need to determine MetS status
mets_criteria = mets_criteria.dropna()

In [ ]:
## criteria for Mets

# fasting glucose > 100 mg/dl
mets_criteria['mets_glc'] = np.where(mets_criteria['glucose_calibrated'] >= 100, 1, 0)

# elevated triglycerides (>= 150 mg/dl)
mets_criteria['mets_tg'] = np.where(mets_criteria['tg_mg_dL'] >= 150, 1, 0)

# split into males and females for sex specific markers 
mets_f = mets_criteria[mets_criteria['Sex']=='Female']
mets_m = mets_criteria[mets_criteria['Sex']=='Male']
mets_f = mets_f.copy()
mets_m = mets_m.copy()

# waist circumference
mets_f['mets_wc'] = np.where(mets_f['WC'] >= 88, 1, 0)
mets_m['mets_wc'] = np.where(mets_m['WC'] >= 102, 1, 0)

# HDL for females
mets_f['mets_hdl'] = np.where(mets_f['hdl_mg_dL'] < 50, 1, 0)

# HDL for males
mets_m['mets_hdl'] = np.where(mets_m['hdl_mg_dL'] < 40, 1, 0)

# recombine males and females
mets_fm = pd.concat([mets_f, mets_m])

In [ ]:
# convery hyptertension to int for determining MetS
mets_fm['mets_htn'] = np.where(mets_fm['hypertension']=='yes', 1, 0)

In [ ]:
mets_criteria['mets_count'] = mets_fm['mets_tg'] + mets_fm['mets_glc'] + mets_fm['mets_wc'] + mets_fm['mets_hdl'] + mets_fm['mets_htn']

In [ ]:
mets_criteria['mets_count'].value_counts()

In [ ]:
# if a participant has three or more of the criteria, they have Mets
mets_criteria['mets'] = np.where(mets_criteria['mets_count'] >= 3, 1, 0)

# select the features for merging
mets_criteria = mets_criteria[['SEQN', 'mets']]

In [ ]:
mets_criteria['mets'].value_counts()

In [ ]:
# match the glycan intake data to the MetS targets
mets_wweia = mets_wweia[mets_wweia['SEQN'].isin(mets_criteria['SEQN'])]

In [ ]:
mets_wweia = mets_wweia[['SEQN', 'Sex', 'Age', 'BMI', 'ethnicity', 'family_pir', 'education',
       'ever_smoker', 'diet_wts_combined',
       'Fiber, total dietary', 'Arabinan', 'Galactan', 'Xyloglucan', 'Xylan',
       'Mannan', 'Beta-Glucan', 'Chitin', 'Starch', 'Cellulose',
       'Arabinoxylan', 'Galactomannan', 'Free_Fructose', 'Free_Glucose',
       'Free_Sucrose', 'Free_Maltose', 'Free_Lactose(trehalose)',
       'Free_Kestose', 'Free_Raffinose', 'Free_Maltotetraose',
       'Free_Stachyose', 'Free_Maltopentaose', 'Free_Verbascose',
       'Free_Maltohexaose', 'Mono_Glucose', 'Mono_Galactose', 'Mono_Fructose',
       'Mono_Xylose', 'Mono_Arabinose', 'Mono_Fucose', 'Mono_Rhamnose',
       'Mono_GalA', 'Mono_Mannose', 'Mono_Ribose']]

### save the MetS target and glycan intake data

In [ ]:
# make the directory
os.makedirs('../../data/01/mets/', exist_ok=True)

# save the ir outcome
mets_criteria.to_csv('../../data/01/mets/mets_target.tsv', sep='\t', index=None)

# save the ir target features
mets_wweia.to_csv('../../data/01/mets/mets_dfg2_intake.tsv', sep='\t', index=None)